# DeepGuard AI - Notebook 04: XceptionNet Forensic Classifier Training
### Project: Deepfake Image & Video Detection System
**Objective:** Fine-tune the Xception architecture (via `timm`) on facial crops.
Xception utilizes depthwise separable convolutions and 299x299 resolution, providing strong sensitivity to high-frequency facial boundary artifacts.

### Step 1: Install timm & Verify GPU

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

!pip install timm scikit-learn matplotlib
import torch
import torch.nn as nn
import timm
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using computing device:", device)

---
### Step 1B: Direct URL Dataset Downloader (Zero Manual Upload)
Optionally download the deepfake dataset directly in Colab without downloading to your PC or uploading to Google Drive.


In [ ]:
# Clone repository if needed to load downloader module
import os
if not os.path.exists('Deep-Learning'):
    !git clone https://github.com/AliKhan-Devs/Deep-Learning.git
%cd Deep-Learning/07_DeepGuard_AI/training
import sys
sys.path.append(os.path.abspath('src'))

from datasets.downloader import download_openrl_deepfake, build_manifest_from_folders
DATA_DIR = '/content/data'
print('Downloading OpenRL/DeepFakeFace directly via URL...')
real_dir, fake_dir = download_openrl_deepfake(DATA_DIR)
manifest_df = build_manifest_from_folders(real_dir, fake_dir, os.path.join(DATA_DIR, 'manifest.csv'), max_per_class=3000)
print('>>> Dataset ready for training:', len(manifest_df), 'samples')


### Step 2: Genuine Xception Architecture
We instantiate genuine Xception via `timm.create_model('xception', pretrained=True, num_classes=2)`.
Input resolution is standard **299x299** with range `[-1, 1]` (`mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]`).

In [ ]:
class XceptionDetector(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, dropout=0.4):
        super().__init__()
        try:
            self.model = timm.create_model("xception", pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
        except Exception:
            self.model = timm.create_model("legacy_xception", pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)

    def forward(self, x):
        return self.model(x)

model = XceptionDetector().to(device)
print(f"Xception initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")

### Step 3: Xception 299x299 Transforms

In [ ]:
xception_train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

xception_val_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

### Step 4: Export Checkpoint to Local Directory
After training, download `best_model.pth` and place into `07_DeepGuard_AI/backend/artifacts/models/xception/`.

In [ ]:
EXPORT_DIR = "/content/drive/MyDrive/DeepGuard_Models/xception"
os.makedirs(EXPORT_DIR, exist_ok=True)

xception_config = {
    "model_name": "xception",
    "display_name": "XceptionNet Forensic Classifier",
    "architecture": "xception",
    "num_classes": 2,
    "classes": ["REAL", "FAKE"],
    "positive_class": "FAKE",
    "version": "1.0.0",
    "input_size": [299, 299],
    "mean": [0.5, 0.5, 0.5],
    "std": [0.5, 0.5, 0.5],
    "decision_threshold": 0.50,
    "preprocessing": {"face_crop_margin": 0.30, "interpolation": "bilinear", "color_space": "RGB"}
}

with open(os.path.join(EXPORT_DIR, "model_config.json"), "w") as f:
    json.dump(xception_config, f, indent=2)

print(f"Xception config saved to {EXPORT_DIR}/model_config.json")
# from google.colab import files
# files.download(os.path.join(EXPORT_DIR, "best_model.pth"))
